In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install neo4j


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
from neo4j import GraphDatabase
import joblib

In [ ]:
model_path = "/content/drive/MyDrive/banking-ai-knowledge-system/nlp_model/intent_model.pkl"
vectorizer_path = "/content/drive/MyDrive/banking-ai-knowledge-system/nlp_model/vectorizer.pkl"

model = joblib.load(model_path)
vectorizer = joblib.load(vectorizer_path)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
uri = "neo4j+s://YOUR_DB_ID.databases.neo4j.io"
username = "neo4j"
password = "YOUR_PASSWORD"

In [ ]:
intent_mapping = {
    "lost_card": "LostCard",
    "activate_card": "ActivateCard",
    "close_account": "CloseAccount"
}

In [ ]:
def get_graph_response(intent_name):

    query = """
    MATCH (i:Intent {name:$intent})-[:TRIGGERS]->(s)-[:REQUIRES]->(v)
    RETURN s.name AS service, v.name AS verification
    """

    with driver.session() as session:
        result = session.run(query, intent=intent_name)
        return [record.data() for record in result]

In [ ]:
def process_query(user_query):

    # Step 1 — Predict intent
    cleaned = clean_text(user_query)
    vec = vectorizer.transform([cleaned])
    predicted_intent = model.predict(vec)[0]

    print("Predicted intent:", predicted_intent)

    # Step 2 — Map to graph node
    graph_intent = intent_mapping.get(predicted_intent)

    if not graph_intent:
        return "Intent not mapped to graph"

    # Step 3 — Query graph
    result = get_graph_response(graph_intent)

    return result

In [ ]:
process_query("My credit card was stolen")

Predicted intent: lost_or_stolen_card


'Intent not mapped to graph'

In [ ]:
def generate_response(user_query):

    result = process_query(user_query)

    if isinstance(result, str):
        return result

    if not result:
        return "No service flow found."

    service = result[0]['service']
    verification = result[0]['verification']

    return f"To resolve your issue, we will initiate {service}. Please complete {verification}."

In [ ]:
generate_response("My card was stolen")

Predicted intent: lost_or_stolen_card


'Intent not mapped to graph'

In [ ]:
def process_query_debug(user_query):

    print("\n--- DEBUG START ---")

    # Step 1 — Predict intent
    cleaned = clean_text(user_query)
    print("Cleaned query:", cleaned)

    vec = vectorizer.transform([cleaned])
    predicted_intent = model.predict(vec)[0]
    print("Predicted intent:", predicted_intent)

    # Step 2 — Mapping
    graph_intent = normalize_intent(predicted_intent)
    print("Mapped graph intent:", graph_intent)

    if not graph_intent:
        print("❌ Mapping failed")
        return None

    # Step 3 — Query graph
    result = get_graph_response(graph_intent)
    print("Graph result:", result)

    print("--- DEBUG END ---\n")

    return result

In [ ]:
process_query_debug("My card was stolen")


--- DEBUG START ---
Cleaned query: card stolen
Predicted intent: lost_or_stolen_card
Mapped graph intent: LostCard
Graph result: [{'service': 'CardReplacement', 'verification': 'IdentityVerification'}]
--- DEBUG END ---



[{'service': 'CardReplacement', 'verification': 'IdentityVerification'}]

In [ ]:
def generate_response(user_query):

    result = process_query_debug(user_query)

    # Case 1 — No result
    if not result:
        return "No service flow found for this query."

    # Case 2 — Extract safely
    try:
        service = result[0].get('service', 'Unknown Service')
        verification = result[0].get('verification', 'Unknown Verification')

        response = f"To resolve your issue, we will initiate {service}. Please complete {verification}."

        return response

    except Exception as e:
        return f"Error generating response: {str(e)}"

In [ ]:
generate_response("My card was stolen")


--- DEBUG START ---
Cleaned query: card stolen
Predicted intent: lost_or_stolen_card
Mapped graph intent: LostCard
Graph result: [{'service': 'CardReplacement', 'verification': 'IdentityVerification'}]
--- DEBUG END ---



'To resolve your issue, we will initiate CardReplacement. Please complete IdentityVerification.'